# Import libraries

In [39]:
import polars as pl
from datetime import datetime

# Load dataset

In [40]:
df = pl.read_csv("../hospital_readmission_dataset.csv")
df.head()

patient_id,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,readmission_risk_score,label
str,str,str,i64,str,str,str,i64,i64,str,i64,i64,i64,str,str,f64,i64
"""P00001""","""2022-04-14""","""Spring""",66,"""Male""","""South""","""Diabetes""",5,6,"""Interventional""",8,6,1,"""Medicare""","""Home Health""",0.92,1
"""P00002""","""2021-09-19""","""Fall""",55,"""Male""","""South""","""Diabetes""",4,6,"""Interventional""",6,4,3,"""Private""","""Home Health""",0.88,1
"""P00003""","""2023-04-12""","""Spring""",69,"""Female""","""West""","""Hypertension""",6,8,"""Medical""",9,6,2,"""Medicare""","""Skilled Nursing""",0.97,1
"""P00004""","""2023-08-14""","""Summer""",83,"""Male""","""South""","""Stroke""",6,11,"""Medical""",11,4,2,"""Medicare""","""Skilled Nursing""",0.97,1
"""P00005""","""2021-11-05""","""Fall""",54,"""Female""","""North""","""Stroke""",4,10,"""Medical""",6,2,1,"""Uninsured""","""Home Health""",0.83,1


# Drop unneeded columns

In [41]:
df = df.drop("readmission_risk_score")
df = df.drop("patient_id")

In [42]:
df.describe()

statistic,admission_date,season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,label
str,str,str,f64,str,str,str,f64,f64,str,f64,f64,f64,str,str,f64
"""count""","""8000""","""8000""",8000.0,"""8000""","""8000""","""8000""",8000.0,8000.0,"""8000""",8000.0,8000.0,8000.0,"""8000""","""8000""",8000.0
"""null_count""","""0""","""0""",0.0,"""0""","""0""","""0""",0.0,0.0,"""0""",0.0,0.0,0.0,"""0""","""0""",0.0
"""mean""",null,null,57.411625,null,null,null,4.3185,7.789125,null,7.47575,3.638125,1.5725,null,null,0.772875
"""std""",null,null,16.724388,null,null,null,1.358046,1.930252,null,2.287917,1.630415,0.89196,null,null,0.419
"""min""","""2021-01-01""","""Fall""",18.0,"""Female""","""Central""","""Appendicitis""",1.0,3.0,"""Conservative""",2.0,0.0,0.0,"""Medicaid""","""Home""",0.0
"""25%""",null,null,46.0,null,null,null,3.0,6.0,null,6.0,2.0,1.0,null,null,1.0
"""50%""",null,null,57.0,null,null,null,4.0,8.0,null,8.0,4.0,1.0,null,null,1.0
"""75%""",null,null,69.0,null,null,null,5.0,9.0,null,9.0,4.0,2.0,null,null,1.0
"""max""","""2023-12-31""","""Winter""",95.0,"""Male""","""West""","""Stroke""",10.0,15.0,"""Surgical""",18.0,10.0,5.0,"""Uninsured""","""Skilled Nursing""",1.0


# Convert date to week day

In [43]:
df = df.with_columns(
    pl.col("admission_date").str.to_date().dt.to_string("%A").alias("admission_weekday")
).drop("admission_date")

In [44]:
df['admission_weekday'].head()

admission_weekday
str
"""Thursday"""
"""Sunday"""
"""Wednesday"""
"""Monday"""
"""Friday"""
"""Thursday"""
"""Tuesday"""
"""Saturday"""
"""Monday"""


# One-Hot encoding for categorical variables

In [45]:
df.head()

season,age,gender,region,primary_diagnosis,comorbidities_count,length_of_stay,treatment_type,medications_count,followup_visits_last_year,prev_readmissions,insurance_type,discharge_disposition,label,admission_weekday
str,i64,str,str,str,i64,i64,str,i64,i64,i64,str,str,i64,str
"""Spring""",66,"""Male""","""South""","""Diabetes""",5,6,"""Interventional""",8,6,1,"""Medicare""","""Home Health""",1,"""Thursday"""
"""Fall""",55,"""Male""","""South""","""Diabetes""",4,6,"""Interventional""",6,4,3,"""Private""","""Home Health""",1,"""Sunday"""
"""Spring""",69,"""Female""","""West""","""Hypertension""",6,8,"""Medical""",9,6,2,"""Medicare""","""Skilled Nursing""",1,"""Wednesday"""
"""Summer""",83,"""Male""","""South""","""Stroke""",6,11,"""Medical""",11,4,2,"""Medicare""","""Skilled Nursing""",1,"""Monday"""
"""Fall""",54,"""Female""","""North""","""Stroke""",4,10,"""Medical""",6,2,1,"""Uninsured""","""Home Health""",1,"""Friday"""


In [46]:
# One-Hot Encoding

In [47]:
df.to_dummies(["season", "gender", "region", "primary_diagnosis", "treatment_type", "insurance_type", "discharge_disposition", "admission_weekday"], drop_first=True)

season_Fall,season_Summer,season_Winter,age,gender_Female,region_Central,region_East,region_North,region_West,primary_diagnosis_Appendicitis,primary_diagnosis_COPD,primary_diagnosis_Fracture,primary_diagnosis_Heart Failure,primary_diagnosis_Hypertension,primary_diagnosis_Influenza,primary_diagnosis_Kidney Disease,primary_diagnosis_Pneumonia,primary_diagnosis_Sepsis,primary_diagnosis_Stroke,comorbidities_count,length_of_stay,treatment_type_Conservative,treatment_type_Medical,treatment_type_Surgical,medications_count,followup_visits_last_year,prev_readmissions,insurance_type_Medicaid,insurance_type_Private,insurance_type_Uninsured,discharge_disposition_Home,discharge_disposition_Rehab,discharge_disposition_Skilled Nursing,label,admission_weekday_Friday,admission_weekday_Monday,admission_weekday_Saturday,admission_weekday_Sunday,admission_weekday_Tuesday,admission_weekday_Wednesday
u8,u8,u8,i64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,i64,i64,u8,u8,u8,i64,i64,i64,u8,u8,u8,u8,u8,u8,i64,u8,u8,u8,u8,u8,u8
0,0,0,66,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,5,6,0,0,0,8,6,1,0,0,0,0,0,0,1,0,0,0,0,0,0
1,0,0,55,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,4,6,0,0,0,6,4,3,0,1,0,0,0,0,1,0,0,0,1,0,0
0,0,0,69,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,6,8,0,1,0,9,6,2,0,0,0,0,0,1,1,0,0,0,0,0,1
0,1,0,83,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,6,11,0,1,0,11,4,2,0,0,0,0,0,1,1,0,1,0,0,0,0
1,0,0,54,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,4,10,0,1,0,6,2,1,0,0,1,0,0,0,1,1,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,0,1,95,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,6,9,0,0,0,11,6,2,0,0,0,0,0,1,1,0,0,0,0,1,0
0,0,0,63,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,4,7,0,0,1,7,4,1,1,0,0,0,0,0,1,0,0,0,0,0,0
0,1,0,43,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,3,6,0,1,0,4,2,3,0,0,1,0,0,0,1,0,0,0,0,0,0


In [52]:
df.write_parquet("01_preprocessing/data.parquet")

FileNotFoundError: No such file or directory (os error 2): 01_preprocessing/data.parquet